In [1]:

import os
import pandas as pd
import torch
from torchvision.io import decode_image, ImageReadMode
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split


In [2]:

class IsingDataset(Dataset):
    def __init__(self, csv_path, data_root, split=None, transform=None):
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        self.df        = df.reset_index(drop=True)
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.data_root, row["filepath"])
        image    = decode_image(img_path, mode=ImageReadMode.GRAY)  # → uint8 [1, H, W]
        label    = int(row["label"])
        T        = float(row["T"])
        L        = int(row["L"])
        if self.transform:
            image = self.transform(image)
        return image, label, T, L


In [3]:

# Transform pipeline:
#   ImageReadMode.GRAY already gives [1, H, W] uint8 — no Grayscale() needed
#   Resize    : → [1, 64, 64]
#   float32   : uint8 [0,255]  → float [0.0, 1.0]
#   Normalize : [0,1]          → [-1, 1]  (aligns with physical spin values ±1)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize((0.5,), (0.5,)),
])

DATA_ROOT = "data"
CSV_PATH  = "data/dataset.csv"

full_dataset = IsingDataset(CSV_PATH, DATA_ROOT, split="train",    transform=transform)
critical_set = IsingDataset(CSV_PATH, DATA_ROOT, split="critical", transform=transform)

n       = len(full_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

print(f"train: {len(train_set)}  |  val: {len(val_set)}  |  test: {len(test_set)}  |  critical: {len(critical_set)}")


train: 3301  |  val: 707  |  test: 708  |  critical: 2352


In [4]:

train_loader    = DataLoader(train_set,    batch_size=64, shuffle=True,  num_workers=0)
val_loader      = DataLoader(val_set,      batch_size=64, shuffle=False, num_workers=0)
test_loader     = DataLoader(test_set,     batch_size=64, shuffle=False, num_workers=0)
critical_loader = DataLoader(critical_set, batch_size=64, shuffle=False, num_workers=0)

# sanity check
imgs, labels, temps, sizes = next(iter(train_loader))
print(f"image batch : {imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"labels      : {labels[:8].tolist()}")
print(f"temperatures: {[f'{t:.4f}' for t in temps[:8].tolist()]}")
print(f"lattice sizes: {sizes[:8].tolist()}")


image batch : torch.Size([64, 1, 64, 64])  dtype=torch.float32  range=[-1.00, 1.00]
labels      : [0, 0, 0, 1, 1, 1, 0, 1]
temperatures: ['1.5916', '1.0030', '1.6186', '2.8739', '3.5465', '3.5165', '1.6126', '2.7958']
lattice sizes: [10, 10, 50, 10, 10, 10, 60, 20]
